<a href="https://colab.research.google.com/github/vanshg27/vanshg27/blob/main/MIamiGPipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Predicting the Miami GP Winner with Machine Learning

This notebook aims to develop a machine learning model to predict the winner of the upcoming Miami Grand Prix.

In [ ]:
pip install fastf1

In [ ]:
import fastf1
import pandas as pd
import os

# Configure cache for better performance
cache_dir = './cache'
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
fastf1.Cache.enable_cache(cache_dir)

print("FastF1 library imported and cache enabled.")

FastF1 library imported and cache enabled.


### Identify Last Race and Fetch Reference Data
We will find the most recent completed race in 2024 to use as a reference for our prediction model.

In [ ]:
import datetime
import fastf1
import pandas as pd

# List of races to include for training
races_to_load = [
    (2026, 'Australia', 'R'),
    (2026, 'China', 'R'),
    (2026, 'Japan', 'R')
]

all_results = []

for year, loc, sess in races_to_load:
    print(f"Loading {year} {loc} Grand Prix...")
    session = fastf1.get_session(year, loc, sess)
    session.load()
    res = session.results.copy()
    # Add a column to identify the event
    res['Event'] = loc
    all_results.append(res)

# Combine all reference data into one DataFrame
ref_data_combined = pd.concat(all_results, ignore_index=True)

# We'll keep the most recent race (Japan) as the reference for the Miami entry list
ref_data = all_results[-1]

print("Aggregated data from 3 races loaded.")
display(ref_data_combined[['Event', 'FullName', 'ClassifiedPosition', 'GridPosition']].head())

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


Loading 2026 Australia Grand Prix...


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

Loading 2026 China Grand Prix...


core        WARNING 	Driver 12 completed the race distance 00:00.022000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
INFO:fastf1.fastf1.req:Using cached data for car_data
req            INFO 	Using cached data for position_data
INFO:fastf1.fastf1.req:Using cached data for position_data
req            INFO 	Using cached data for weather_data
INFO:fastf1.fastf1.req:Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
INFO:fastf1.fastf1.req:Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '87', '10', '30', '6', '55', '43', '27', '41', '77', '31', '11', '3', '14', '18', '81', '1', '5', '23']
INFO:fastf1.fastf1.core:Finished loading data for 22 drivers: ['12', '63', '44', '16', '87', '10', '30', '6', '55', '43', '27', '41', '77', '31', '11', '3', '14', '18', '81', '1', '5', '23']
core           INFO 	Loading data for Jap

Loading 2026 Japan Grand Prix...


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
INFO:fastf1.api:Fetching lap count data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for track_status_data. Loading data...
_api

Aggregated data from 3 races loaded.


,Event,FullName,ClassifiedPosition,GridPosition
0,Australia,George Russell,1,1.0
1,Australia,Kimi Antonelli,2,2.0
2,Australia,Charles Leclerc,3,4.0
3,Australia,Lewis Hamilton,4,7.0
4,Australia,Lando Norris,5,6.0


### 4. Gradient Boosting Regression Prediction
We will now use a `GradientBoostingRegressor` to predict the race outcome for the Miami GP based on the Chinese GP performance.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np
import pandas as pd

# Prepare training data from the combined 2026 races
train_df = ref_data_combined.copy()
train_df['ClassifiedPosition'] = pd.to_numeric(train_df['ClassifiedPosition'], errors='coerce')
# Ignore DNFs for training
train_df = train_df.dropna(subset=['ClassifiedPosition'])

# Use GridPosition as the feature
X_train = train_df[['GridPosition']].fillna(20)
y_train = train_df['ClassifiedPosition']

# Initialize and train the model on the larger dataset
model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Define upcoming Miami GP
schedule_2026 = fastf1.get_event_schedule(2026)
upcoming_2026 = schedule_2026[schedule_2026['EventDate'] >= datetime.datetime.now()]
next_race_2026 = upcoming_2026.iloc[0]

# Predict using the latest available entry list (from the Japanese GP reference)
X_test = ref_data[['GridPosition']].fillna(20)
predictions = model.predict(X_test)

# Create a prediction dataframe
predict_df = ref_data[['FullName', 'TeamName', 'GridPosition']].copy()
predict_df['PredictedPosition'] = predictions
predict_df = predict_df.sort_values(by='PredictedPosition')

print(f"Gradient Boosting results (Trained on AUS, CHN, JPN) for {next_race_2026['EventName']}:")
display(predict_df.head(10))

Gradient Boosting results (Trained on AUS, CHN, JPN) for Miami Grand Prix:


,FullName,TeamName,GridPosition,PredictedPosition
12,Kimi Antonelli,Mercedes,1.0,1.015687
81,Oscar Piastri,McLaren,3.0,2.543898
63,George Russell,Mercedes,2.0,2.646656
16,Charles Leclerc,Ferrari,4.0,3.336583
1,Lando Norris,McLaren,5.0,5.002604
44,Lewis Hamilton,Ferrari,6.0,5.494092
10,Pierre Gasly,Alpine,7.0,5.670065
30,Liam Lawson,Racing Bulls,14.0,8.690475
31,Esteban Ocon,Haas F1 Team,12.0,9.037386
41,Arvid Lindblad,Racing Bulls,10.0,9.379327


In [25]:
from sklearn.metrics import mean_squared_error

# Generate predictions for the entire combined training dataset
X_combined_test = train_df[['GridPosition']].fillna(20)
predictions_combined = model.predict(X_combined_test)

# Get the actual classified positions for the combined data
y_true_combined = pd.to_numeric(train_df['ClassifiedPosition'], errors='coerce')

# Assign a numerical value for DNF positions in the combined data (for general context, not used for top 3 specific MSE)
dnf_position_value_combined = len(train_df) + 1
y_true_filled_combined = y_true_combined.fillna(dnf_position_value_combined)

# Calculate Mean Squared Error for each of the top 3 positions separately
for position in [1, 2, 3]:
    # Filter for the specific position
    target_position_indices = (y_true_filled_combined == position) | (predictions_combined.round() == position)

    # Only calculate if there are actual values for this position
    if target_position_indices.any():
        y_true_at_position = y_true_filled_combined[target_position_indices]
        predictions_at_position = predictions_combined[target_position_indices]

        mse_at_position = mean_squared_error(y_true_at_position, predictions_at_position)
        print(f"Mean Squared Error for {position}st/nd/rd position predictions across all races: {mse_at_position:.2f}")
    else:
        print(f"No data found for {position}st/nd/rd position in predictions or actuals.")

Mean Squared Error for 1st/nd/rd position predictions across all races: 0.00
Mean Squared Error for 2st/nd/rd position predictions across all races: 0.38
Mean Squared Error for 3st/nd/rd position predictions across all races: 0.48
